# Agent for Amazon Bedrock with Code Interpreter Overview

This is the final notebook in the series to demonstrates how to set up and use an Amazon Bedrock Agent with Code Interpreter capabilities.

In this notebook, we'll walk through the process of testing and cleaning up an Agent in Amazon Bedrock. We'll see how to set up the Code Interpreter action.  Code Interpreter enables your agent to write and execute code, process documents, and respond to complex queries via access to a secure code execution sandbox.

_(Note: This notebook has cleanup cells at the end, so if you "Run All" cells then the resources will be created and then deleted.)_

**Note:** At the time of writing Code Interpreter is in public preview.  

## Step 1: Import Required Libraries

First, we need to import the necessary Python libraries. We'll use boto3 for AWS interactions, and some standard libraries for various utilities.

In [1]:
import os
import io
import time
import json
import boto3
import logging
import uuid, string
import time, random 
import pandas as pd
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt

In [2]:
# set a logger
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

## Step 2: Set the AWS Region

We're using the US East (N. Virginia) region for this demo. Feel free to change this to your preferred region, but make sure that a) the region supports Amazon Bedrock, b) Agents, c) the Claude Sonnet (3) model, and finally d) you have enabled access to the Sonnet (3) in this region. 

In [3]:
region_name: str = 'us-east-1'

In [4]:
# Read the S3 URI from the text file
with open('s3_uri.txt', 'r') as f:
    s3_uri = f.read().strip()

print(f"Loaded S3 URI: {s3_uri}") 

Loaded S3 URI: s3://sagemaker-us-east-1-015469603702/code-interpreter-demo-data/nyc_taxi_subset.csv


In [5]:
# constants
CSV_DATA_FILE: str = 'nyc_taxi_subset.csv'
# Bucket and prefix name where this csv file will be uploaded and used as S3 source by code interpreter
S3_BUCKET_NAME: str = s3_uri.replace("s3://", "")
PREFIX: str = 'code-interpreter-demo-data'
# This is the size of the file that will be uploaded to s3 and used by the agent (in MB)
DATASET_FILE_SIZE: float = 99

In [6]:
# Read the agent info from the JSON file
with open('agent_info.json', 'r') as f:
    agent_info = json.load(f)

# Extract the agent information
agentId = agent_info['agentId']
agentAliasId = agent_info['agentAliasId']
agentAliasStatus = agent_info['agentAliasStatus']
role_name = agent_info['role_name']

print(f"Loaded agent information:")
print(f"agentId: {agentId}")
print(f"agentAliasId: {agentAliasId}")
print(f"agentAliasStatus: {agentAliasStatus}")
print(f"roleName: {role_name}")


Loaded agent information:
agentId: 926JKYCBP0
agentAliasId: THBZNJNOHE
agentAliasStatus: PREPARED
roleName: test-agent-OMYGJ


In [7]:
from botocore.config import Config
custom_config = Config(
            read_timeout=300,  # 5 minutes
            connect_timeout=10,  # 10 seconds
            retries={'max_attempts': 3}
        )

In [8]:
bedrock_agent_runtime = boto3.client(service_name = 'bedrock-agent-runtime', region_name = region_name, config=custom_config)

[2024-11-12 11:02:47,284] p49986 {credentials.py:1278} INFO - Found credentials in shared credentials file: ~/.aws/credentials


## Step 3: Implement Agent Interaction Function

Let's now develop a function that facilitates communication with our agent. This function will be responsible for:
1. Sending user messages to the agent
2. Receiving the agent's responses
3. Processing and presenting the returned information

This encapsulation will streamline our interaction process and make it easier to engage with the agent throughout our session.

In [9]:
import pprint
def process_response(resp, enable_trace:bool=False, show_code_use:bool=False):
    if enable_trace:
        logger.info(pprint.pprint(resp))

    event_stream = resp['completion']
    try:
        for event in event_stream:
            if 'chunk' in event:
                data = event['chunk']['bytes']
                if enable_trace:
                    logger.info(f"Final answer ->\n{data.decode('utf8')}")
                agent_answer = data.decode('utf8')
                return agent_answer
                # End event indicates that the request finished successfully
            elif 'trace' in event:
                if 'codeInterpreterInvocationInput' in json.dumps(event['trace']):
                    if show_code_use:
                        print("Invoked code interpreter")
                if enable_trace:
                    logger.info(json.dumps(event['trace'], indent=2))
            else:
                raise Exception("unexpected event.", event)
    except Exception as e:
        raise Exception("unexpected event.", e)

In [10]:
def invoke_agent_helper(
    query, session_id, agent_id, alias_id, enable_trace=False, memory_id=None, session_state=None, end_session=False, show_code_use=False
):
    
    if not session_state:
        session_state = {}

    # invoke the agent API
    agent_response = bedrock_agent_runtime.invoke_agent(
        inputText=query,
        agentId=agent_id,
        agentAliasId=alias_id,
        sessionId=session_id,
        enableTrace=(enable_trace | show_code_use), # Force tracing on if showing code use
        endSession=end_session,
        memoryId=memory_id,
        sessionState=session_state
    )
    return process_response(agent_response, enable_trace=enable_trace, show_code_use=show_code_use)

In [11]:
def add_file_to_session_state(use_case='CHAT', session_state=None):
    if use_case != "CHAT" and use_case != "CODE_INTERPRETER":
        raise ValueError("Use case must be either 'CHAT' or 'CODE_INTERPRETER'")
    if not session_state:
        session_state = {
            "files": []
        }
    # Parse S3 URI to get bucket name and key
    s3_client = boto3.client('s3')
    s3_parts = s3_uri.replace("s3://", "").split("/", 1)
    bucket_name = s3_parts[0]
    object_key = s3_parts[1]

    # Extract file name and determine the file type
    file_name = object_key.split("/")[-1]
    file_type = file_name.split(".")[-1].upper()

    # Determine media type based on file extension
    if file_type == "CSV":
        media_type = "text/csv"
    elif file_type in ["XLS", "XLSX"]:
        media_type = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
    else:
        media_type = "text/plain"

    # Download the file locally
    local_file_path = f"/tmp/{file_name}"
    s3_client.download_file(bucket_name, object_key, local_file_path)

    # Read the downloaded file as bytes
    with open(local_file_path, "rb") as f:
        file_data = f.read()
        logger.info(f"Read the s3 file data: {file_data}")

    # Create the named file structure with byte content
    named_file = {
        "name": os.path.basename(CSV_DATA_FILE),
        # "source": { 
        #     "sourceType": "S3",
        #     "s3Location": {
        #         "uri": s3_uri 
        #     }
        # },
        "source": {
            "sourceType": "BYTE_CONTENT",
            "byteContent": {
                "mediaType": 'text/csv',
                "data": file_data
            }
        },
        "useCase": use_case
    }

    # Append to session state
    session_state['files'].append(named_file)

    return session_state

In [12]:
session_id:str = str(uuid.uuid1())
memory_id:str = 'TST_MEM_ID'
sessionState=add_file_to_session_state('CHAT')
query = """Generate code for creating an ML model using SageMaker AutoML to predict total_amount for a trip given the passenger_count and trip_distance from the provided nyc taxi dataset.
            Use the s3_uri you have access to as input data to use. Generate accurate code that I can use to directly execute. Do not ask a follow up question, just provide the code."""

invoke_agent_helper(query, session_id, agentId, agentAliasId, enable_trace=True, session_state=sessionState, 
                    memory_id=memory_id, show_code_use=True)

[2024-11-12 11:02:54,560] p49986 {1422547881.py:33} INFO - Read the s3 file data: b'VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee\n2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0\n1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.8,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0\n1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.7,1.0,N,236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0\n1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.4,1.0,N,79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0\n1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.8,1.0,N,211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0\n1,2024-01-01 00:54:08,2024-01-01 01:26:31,1.0,4.7,1.0,N,148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0\n2,2024-01-01 00:4

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/vnd.amazon.eventstream',
                                      'date': 'Tue, 12 Nov 2024 16:02:54 GMT',
                                      'transfer-encoding': 'chunked',
                                      'x-amz-bedrock-agent-memory-id': 'TST_MEM_ID',
                                      'x-amz-bedrock-agent-session-id': '93848ea8-a10f-11ef-b7be-168aeb12c116',
                                      'x-amzn-bedrock-agent-content-type': 'application/json',
                                      'x-amzn-requestid': '7374c57f-46c5-40a7-b668-16f6df8582b6'},
                      'HTTPStatusCode': 200,
                      'RequestId': '7374c57f-46c5-40a7-b668-16f6df8582b6',
                      'RetryAttempts': 0},
 'completion': <botocore.eventstream.EventStream object at 0x16977cdd0>,
 'contentType': 'application/json',
 'memoryId': 'TST_MEM_ID',
 's

[2024-11-12 11:02:55,068] p49986 {2945040320.py:21} INFO - {
  "agentAliasId": "THBZNJNOHE",
  "agentId": "926JKYCBP0",
  "agentVersion": "1",
  "sessionId": "93848ea8-a10f-11ef-b7be-168aeb12c116",
  "trace": {
    "orchestrationTrace": {
      "modelInvocationInput": {
        "inferenceConfiguration": {
          "maximumLength": 2048,
          "stopSequences": [
            "</invoke>",
            "</answer>",
            "</error>"
          ],
          "temperature": 0.0,
          "topK": 250,
          "topP": 1.0
        },
        "text": "{\"system\":\"You are a machine model code generator for SageMaker autoML. Your role is to carefully analyze the data and generatecode to train ML models based on the user question, creating ML models, and providing natural language summaries of complex analysis. Your primary function is to assist users by solving problems and fulfilling requests through these capabilities.Here are your key attributes and instructions:ML Model Generation:

The `invoke` function is our primary interface for agent interaction. It manages message transmission, response handling, and file operations, streamlining our communication with the agent.


## Step 5: Cleaning Up

Let's delete the agent and its associated resources.

In [15]:
# Set up Bedrock Agent and IAM clients
bedrock_agent = boto3.client(service_name = 'bedrock-agent', region_name = region_name)
iam = boto3.client('iam')

In [16]:
response = bedrock_agent.delete_agent(
    agentId=agentId,
    skipResourceInUseCheck=True
)

response['agentStatus']

'DELETING'

Finally, let's clean up the IAM role and policies we created for this demo.

In [17]:
# List and delete all inline policies
inline_policies = iam.list_role_policies(RoleName=role_name)
for policy_name in inline_policies.get('PolicyNames', []):
    iam.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
    print(f"Deleted inline policy: {policy_name}")

# List and detach all managed policies
attached_policies = iam.list_attached_role_policies(RoleName=role_name)
for policy in attached_policies.get('AttachedPolicies', []):
    iam.detach_role_policy(RoleName=role_name, PolicyArn=policy['PolicyArn'])
    print(f"Detached managed policy: {policy['PolicyName']}")

# Wait a moment to ensure AWS has processed the policy detachments
time.sleep(10)

# Now attempt to delete the role
try:
    iam.delete_role(RoleName=role_name)
    print(f"Successfully deleted role: {role_name}")
except iam.exceptions.DeleteConflictException:
    print(f"Failed to delete role: {role_name}. Please check if all policies are detached.")
except Exception as e:
    print(f"An error occurred while deleting role {role_name}: {str(e)}")

Deleted inline policy: policy-s3-access-SPRMD
Deleted inline policy: policy-test-agent-SPRMD
Successfully deleted role: test-agent-SPRMD


## Next Steps: Bedrock Agent with Code Interpreter

We've just completed a comprehensive journey through the creation and utilization of a Bedrock Agent with Code Interpreter capabilities. This demonstration has illustrated the following key steps:

1. Establishing the required AWS infrastructure for a Bedrock Agent
2. Developing and configuring an agent with Code Interpreter functionality
3. Engaging in a dialogue with the agent and analyzing its outputs

This walkthrough highlights the robust features of Bedrock Agents, showcasing their potential for handling intricate queries and executing code within a controlled environment. The versatility of this technology opens up a wide array of possibilities across various domains and applications.

By mastering these steps, you've gained valuable insights into creating AI-powered assistants capable of tackling complex, code-related tasks. This foundation sets the stage for further exploration and innovative implementations in your projects.
